In [1]:
import torch
import torchnmf
from torchaudio import load
import numpy as np
from scipy.sparse import coo_matrix
import pandas as pd
import scipy.sparse as sp
from nmf import run_nmf
import muon as mu 
import scanpy as sc

torch.set_flush_denormal(True)

torchnmf.__version__

/home/users/ymo/.local/lib/python3.9/site-packages/torch/_subclasses/functional_tensor.py:275: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ModuleNotFoundError: No module named 'numpy'

In [2]:
!lscpu | grep "CPU"
print(torch.cuda.get_device_name())

CPU op-mode(s):        32-bit, 64-bit
CPU(s):                20
On-line CPU(s) list:   0-19
CPU family:            6
Model name:            Intel(R) Xeon(R) CPU E5-2640 v4 @ 2.40GHz
CPU MHz:               2612.548
CPU max MHz:           3400.0000
CPU min MHz:           1200.0000
NUMA node0 CPU(s):     0-9
NUMA node1 CPU(s):     10-19


NVIDIA TITAN V


# Load Perturb-seq data

In [ ]:
# file name
file_name = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/NMF_benchmark_results_7.24.25"

In [3]:
# reading
mdata = mu.read("/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/250_gene_rawcounts_7.16.25.h5mu")    # returns a MuData object
adata = mdata.mod['rna']
cNNMF = mdata.mod['cNMF']

/home/users/ymo/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/users/ymo/.local/lib/python3.9/site-packages/mudata/_core/mudata.py:477: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  warnings.warn(


In [4]:
# shared parameter
max_iter = 500
tol = 1e-4
init = 'random'
beta_loss = 2.0
verbose = True

# Matrix & componments
components = [10, 20, 50, 100, 150, 200]


# Different pacakges

In [7]:
def sk_mu(k, Snumpy):
    
    print("sk_mu started")
    
    #sklearn mu
    start = time.time()
    
    W, H, n_iter = SKNMF(Snumpy, n_components= k, init=init, solver="mu", max_iter=max_iter, beta_loss=beta_loss, tol=tol,
                  random_state = None, verbose=verbose)
    
    total_time = (time.time() - start)
    
    err = explained_variance_score(Snumpy,np.dot(W, H))
    
    np.save(f'{file_name}/H_sk_mu_K{k}_cell{Snumpy.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_sk_mu_K{k}_cell{Snumpy.shape[0]}.npy', W) 
    
    return err, total_time

def sk_cd(k, Snumpy):
    
    print("sk_cd started")
    
    #sklearn cd
    start = time.time()
    
    W, H, n_iter =  SKNMF(Snumpy, n_components= k, init=init, solver="cd", max_iter=max_iter, beta_loss=beta_loss, tol=tol,
                  random_state = None, verbose=verbose)
        
    
    total_time = (time.time() - start)
    
    err = explained_variance_score(Snumpy,np.dot(W, H))
    
    np.save(f'{file_name}/H_sk_cd_K{k}_cell{Snumpy.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_sk_cd_K{k}_cell{Snumpy.shape[0]}.npy', W) 
    
    return err, total_time

def nmftorch_mu(k, Storch , random_state):
    
    print("nmftorch_mu started")
    
    #nmftorch mu
    start = time.time()
    W, H, err = run_nmf(Storch, n_components= k, init=init, algo="mu", beta_loss=beta_loss, tol=tol,
                  random_state=random_state, n_jobs=1)
    total_time = (time.time() - start)
    
    
    err = explained_variance_score(Storch.numpy(),np.dot(W, H))
    
    
    np.save(f'{file_name}/H_nmftorch_mu_K{k}_cell{Storch.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_nmftorch_mu_K{k}_cell{Storch.shape[0]}.npy', W) 
    
    return err,total_time

def nmftorch_halsvar(k, Storch, random_state):
    
    print("nmftorch_halsvar started")
 
    #mnf-torch halsvar
    start = time.time()
    
    W, H, err = run_nmf(Storch, n_components= k, init=init, algo="halsv", beta_loss=beta_loss, tol=tol,
                  random_state=random_state, n_jobs=1)
        
    total_time = (time.time() - start)
    
    err = explained_variance_score(Storch.numpy(),np.dot(W, H))
    
    np.save(f'{file_name}/H_nmftorch_halsvar_K{k}_cell{Storch.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_nmftorch_halsvar_K{k}_cell{Storch.shape[0]}.npy', W) 
    
    return err,total_time

def nmftorch_mu_10jobs(k, Storch, random_state):
    
    print("nmftorch_mu_10jobs started")
    
    #nmftorch mu
    start = time.time()
    W, H, err =run_nmf(Storch, n_components= k, init=init, algo="mu", beta_loss=beta_loss, tol=tol,
                  random_state=random_state, n_jobs=10)
    total_time = (time.time() - start)
    
    err = explained_variance_score(Storch.numpy(),np.dot(W, H))
    
    np.save(f'{file_name}/H_nmftorch_mu_10jobs_K{k}_cell{Storch.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_nmftorch_mu_10jobs_K{k}_cell{Storch.shape[0]}.npy', W) 
    

    return err,total_time

def nmftorch_mu_cuda(k, Scuda, random_state):
    
    print("nmftorch_mu_cuda started")
    
    #nmftorch mu
    start = time.time()
    torch.cuda.synchronize()

    W, H, err = run_nmf(Scuda, n_components= k, init=init, algo="mu", beta_loss= beta_loss, tol=tol,
                  random_state=random_state, use_gpu = True, n_jobs = 1 )
    
    torch.cuda.synchronize()
    total_time = (time.time() - start)
    
    err = explained_variance_score(Scuda.detach().cpu().numpy(),np.dot(W, H))

    
    np.save(f'{file_name}/H_nmftorch_cuda_K{k}_cell{Scuda.shape[0]}.npy', H) 
    np.save(f'{file_name}/W_nmftorch_cuda_K{k}_cell{Scuda.shape[0]}.npy', W) 
    
    
    return err, total_time
          
def torchnmf(k, Storch):
        
    #torchnmf
    net = torchNMF(Storch.shape, rank=k)
    net.cpu()
    
    print("torchnmf started")
    
    start = time.time()
    
    niter = net.fit(Storch, max_iter=max_iter, beta=beta_loss tol=tol, verbose=verbose)
    
    total_time = (time.time() - start)
    
    
    return -10, total_time

def  torchnmf_cuda(k, Scuda):
    
    print("torchnmf_cuda started")
    
    #torchnmf cuda
    net = torchNMF(Scuda.shape, rank=k)
    net.cuda()
    
    torch.cuda.synchronize()
    start = time.time()
    
    niter = net.fit(Scuda, max_iter=max_iter, beta=beta_loss, tol=tol, verbose=verbose)
    
    torch.cuda.synchronize()
    total_time = (time.time() - start)
    
    return -1, total_time

# graph memory
def graph_benchmarking_results(packages, data, file_name, components = ([10,20,50,100,150,200])*6, fig_name = "Log Runtime(s)"):

    # put data in tidy form: one row per observation
    data = {
        "component":  components,
        "package":    package,
        name:    data
    }
    
    df = pd.DataFrame(data)

    # Pivot so each column is a package, index is component
    pivot = df.pivot(index="component", columns="package", values=name)

    # Plot: group-by-component on X, one bar per package
    fig, ax = plt.subplots(figsize=(8, 5))

    # width of each bar (as fraction of 1 component “slot”)
    total_packages = len(pivot.columns)
    bar_width = 0.8 / total_packages               # leave a little padding

    for i, pkg in enumerate(pivot.columns):
        # compute bar positions for this package
        x_pos = np.arange(len(pivot.index)) + (i - total_packages/2) * bar_width + bar_width/2
        ax.bar(x_pos, pivot[pkg], width=bar_width, label=pkg)

    ax.set_xlabel("Component value")
    ax.set_ylabel(fig_name)
    ax.set_title(f"{fig_name} by package across component values")
    ax.set_xticks(np.arange(len(pivot.index)))
    ax.set_xticklabels(pivot.index)
    ax.legend(title="Package", frameon=False)
    ax.grid(axis="y", linestyle=":", alpha=0.4)

    plt.tight_layout()
    plt.show()

    plt.savefig(f"/{file_name}/{name} by package across component values for {Storch.shape[0]} cells.png", dpi=300, bbox_inches='tight')

In [8]:
adata.var_names_make_unique()

adata_sub = sc.pp.subsample(
    adata,
    n_obs=500,               
    copy=True            
)


sk_mu_m = []
sk_cd_m =[]

sk_mu_l = []
sk_cd_l =[]

sk_mu_s = []
sk_cd_s =[]


nmf_torch_mu_m = []
nmf_torch_halsvar_m = []
nmf_torch_mu_cuda_m = []
nmf_torch_mu_10jobs_m = []

nmf_torch_mu_l = []
nmf_torch_halsvar_l = []
nmf_torch_mu_cuda_l = []
nmf_torch_mu_10jobs_l = []

nmf_torch_mu_s = []
nmf_torch_halsvar_s = []
nmf_torch_mu_cuda_s = []
nmf_torch_mu_10jobs_s = []


torchnmf_m = []
torchnmf_cuda_m = []

torchnmf_l = []
torchnmf_cuda_l = []

torchnmf_s = []
torchnmf_cuda_s = []


Snumpy = adata_sub.layers["raw_counts"]
Storch = torch.from_numpy(Snumpy).float()

In [9]:
for k in components:
    
    print('components =', k)
    
    #'''
    peak_MiB, metrics = memory_usage((sk_mu,(k,Snumpy)),max_usage=True,retval=True)
    sk_mu_m.append(peak_MiB)
    sk_mu_l.append(metrics[0])
    sk_mu_s.append(metrics[1])
    
    peak_MiB, metrics = memory_usage((sk_cd,(k,Snumpy)),max_usage=True,retval=True)
    sk_cd_m.append(peak_MiB)
    sk_cd_l.append(metrics[0])
    sk_cd_s.append(metrics[1])
    #'''
    
    peak_MiB, metrics = memory_usage((nmftorch_mu,(k,Storch,i)),max_usage=True,retval=True)
    nmf_torch_mu_m.append(peak_MiB)
    nmf_torch_mu_l.append(metrics[0])
    nmf_torch_mu_s.append(metrics[1])
    
    peak_MiB, metrics = memory_usage((nmftorch_halsvar,(k,Storch,i)),max_usage=True,retval=True)
    nmf_torch_halsvar_m.append(peak_MiB)
    nmf_torch_halsvar_l.append(metrics[0])
    nmf_torch_halsvar_s.append(metrics[1])
    
    peak_MiB, metrics = memory_usage((nmftorch_mu_10jobs,(k,Storch,i)),max_usage=True,retval=True)
    nmf_torch_mu_10jobs_m.append(peak_MiB)
    nmf_torch_mu_10jobs_l.append(metrics[0])
    nmf_torch_mu_10jobs_s.append(metrics[1])
    

    peak_MiB, metrics = memory_usage((nmftorch_mu_cuda,(k,Storch,i)),max_usage=True,retval=True)
    nmf_torch_mu_cuda_m.append(peak_MiB)
    nmf_torch_mu_cuda_l.append(metrics[0])
    nmf_torch_mu_cuda_s.append(metrics[1])
    
    '''
    peak_MiB, metrics = memory_usage((torchnmf,(k,Storch)),max_usage=True,retval=True)
    torchnmf_m.append(peak_MiB)
    torchnmf_l.append(metrics[0])
    torchnmf_s.append(metrics[1])
    
    peak_MiB, metrics = memory_usage((torchnmf_cuda,(k,Scuda)),max_usage=True,retval=True)
    torchnmf_cuda_m.append(peak_MiB)
    torchnmf_cuda_l.append(metrics[0])
    torchnmf_cuda_s.append(metrics[1])
    '''

components = 10
sk_mu started
Epoch 10 reached after 0.719 seconds, error: 649.359924
Epoch 20 reached after 1.379 seconds, error: 616.907288
Epoch 30 reached after 2.039 seconds, error: 608.324524
Epoch 40 reached after 2.699 seconds, error: 604.489319
Epoch 50 reached after 3.359 seconds, error: 602.702759
Epoch 60 reached after 3.850 seconds, error: 601.679443
Epoch 70 reached after 3.868 seconds, error: 600.994873
Epoch 80 reached after 3.885 seconds, error: 600.484436
Epoch 90 reached after 3.903 seconds, error: 600.077332
Epoch 100 reached after 3.920 seconds, error: 599.742737
Epoch 110 reached after 3.937 seconds, error: 599.462036
Epoch 120 reached after 3.955 seconds, error: 599.212952
Epoch 130 reached after 3.973 seconds, error: 598.996826
Epoch 140 reached after 3.990 seconds, error: 598.823486
Epoch 150 reached after 4.007 seconds, error: 598.686340
Epoch 160 reached after 4.025 seconds, error: 598.578003
Epoch 170 reached after 4.042 seconds, error: 598.487854
Epoch 180 

 niter=170, loss=598.2583054166486.
    Converged after 170 iteration(s).
nmftorch_mu_cuda started
Use GPU mode.
 niter=10, loss=638.764628012541.
 niter=20, loss=615.192754346148.
 niter=30, loss=607.9649969365013.
 niter=40, loss=604.517369477503.
 niter=50, loss=602.5933278339547.
 niter=60, loss=601.5517953593023.
 niter=70, loss=600.8338736705846.
 niter=80, loss=600.1891628895011.
 niter=90, loss=599.6101858707872.
 niter=100, loss=599.1693729656081.
 niter=110, loss=598.836423825405.
 niter=120, loss=598.5813697819203.
 niter=130, loss=598.3787732281285.
 niter=140, loss=598.2320828909128.
 niter=150, loss=598.120624330912.
 niter=160, loss=598.0230189967607.
 niter=170, loss=597.9304150568024.
 niter=180, loss=597.8254866597777.
 niter=190, loss=597.7232846393053.
 niter=200, loss=597.6390005680687.
    Converged after 200 iteration(s).
components = 20
sk_mu started
Epoch 10 reached after 0.647 seconds, error: 628.763977
Epoch 20 reached after 1.267 seconds, error: 595.332336
E

violation: 0.002924935708650704
violation: 0.00289157639904786
violation: 0.0028583826803748567
violation: 0.002825153414979481
violation: 0.0027890354316442165
violation: 0.002748872498964211
violation: 0.0027103131510684136
violation: 0.0026725830961281728
violation: 0.002633979322989814
violation: 0.0025942533636545313
violation: 0.002555174328553137
violation: 0.002520272637200472
violation: 0.0024817431237438383
violation: 0.0024440719869247016
violation: 0.002408964229035454
violation: 0.002376441491469056
violation: 0.0023460384353144463
violation: 0.0023169278240380248
violation: 0.002288876531201604
violation: 0.0022632881463938207
violation: 0.0022356871446824055
violation: 0.0022132901308997625
violation: 0.0021909191651285553
violation: 0.002173536377226463
violation: 0.0021571926373193818
violation: 0.0021428145756687595
violation: 0.0021289081037991096
violation: 0.00211859777534283
violation: 0.002113140635343019
violation: 0.0021098381848906663
violation: 0.002107176848

violation: 0.0012156339112532034
violation: 0.001245909909908423
violation: 0.001278090400106522
violation: 0.0013104960521879334
violation: 0.001343922244466757
violation: 0.0013800424474320681
violation: 0.0014174704694778057
violation: 0.001438506034001089
violation: 0.0014709887529434018
violation: 0.0015090545595698923
violation: 0.0015488585323730087
violation: 0.0015928869834217387
violation: 0.0016300401630410502
violation: 0.0016686493873296715
violation: 0.0017110520572937216
violation: 0.0017301081035143702
violation: 0.001753124492005022
violation: 0.0017827347811749986
violation: 0.001811007057767584
violation: 0.0018320100133141454
violation: 0.0018490922535153685
violation: 0.0018735543392823166
violation: 0.001880454418341674
violation: 0.0018898113993718946
violation: 0.0018976076540627876
violation: 0.0018952796538427507
violation: 0.001900633793215889
violation: 0.0019013223028715427
violation: 0.0019027176014891213
violation: 0.0019013989780330245
violation: 0.00189

/home/users/ymo/.local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


...

sk_cd started
violation: 1.0
violation: 0.5646916016569576
violation: 0.2968110589751958
violation: 0.19702202653211956
violation: 0.14841208172185463
violation: 0.12003617522100811
violation: 0.09938152096346807
violation: 0.08527648530028478
violation: 0.07460958307287693
violation: 0.06731976799092146
violation: 0.06144230705641669
violation: 0.05526246574228839
violation: 0.050222833073357565
violation: 0.04558999148702163
violation: 0.04182624514040991
violation: 0.038835928361767975
violation: 0.03683037141320267
violation: 0.03455782726656529
violation: 0.032287375104900595
violation: 0.03024934470791319
violation: 0.028238984204376733
violation: 0.026710369858452952
violation: 0.02497715622160821
violation: 0.023518234564955602
violation: 0.022039632773557315
violation: 0.020590661532534114
violation: 0.01934655986754425
violation: 0.018247520113386522
violation: 0.01714638183510016
violation: 0.016312170267693382
violation: 0.015575395393442769
violation: 0.0147986763575679
vi

violation: 0.000652710703914504
violation: 0.0006344911850118029
violation: 0.0006182073754721396
violation: 0.0006004411188449176
violation: 0.0005808573174460674
violation: 0.0005634457589416974
violation: 0.0005464316755386519
violation: 0.0005300887837032839
violation: 0.0005136535262901948
violation: 0.0004983066381592018
violation: 0.00048218635009078943
violation: 0.00046937811045033714
violation: 0.0004564492880930999
violation: 0.00044549699373639973
violation: 0.00043496705392590803
violation: 0.0004240869201768024
violation: 0.00041274244675393095
violation: 0.0004044831777581309
violation: 0.00039546122009858145
violation: 0.00038615200787638716
violation: 0.00037805484694052705
violation: 0.0003705761726565909
violation: 0.00036309112099270333
violation: 0.00035612107894662814
violation: 0.0003501460444053896
violation: 0.0003452065635416562
violation: 0.0003391643493905156
violation: 0.0003339928732376817
violation: 0.0003279384747998782
violation: 0.00032158492274938697


violation: 0.00021239261951210982
violation: 0.0002102026408600345
violation: 0.00020889803667640806


/home/users/ymo/.local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmftorch_mu started
 niter=10, loss=568.2578365055778.
 niter=20, loss=488.7877223294382.
 niter=30, loss=452.4697503701214.
 niter=40, loss=431.8377878324221.
 niter=50, loss=418.15980796819775.
 niter=60, loss=408.24334042333135.
 niter=70, loss=401.214795963459.
 niter=80, loss=395.9924241699581.
 niter=90, loss=391.81165756010887.
 niter=100, loss=388.35767985196327.
 niter=110, loss=385.52853331498045.
 niter=120, loss=383.202214894434.
 niter=130, loss=381.34334922743835.
 niter=140, loss=379.75699466895935.
 niter=150, loss=378.3639121269363.
 niter=160, loss=377.19391564552046.
 niter=170, loss=376.19127448679615.
 niter=180, loss=375.2896381463256.
 niter=190, loss=374.47388093163454.
 niter=200, loss=373.7582775003117.
 niter=210, loss=373.129334145682.
 niter=220, loss=372.5333877654458.
 niter=230, loss=371.9973118182442.
 niter=240, loss=371.469547877077.
 niter=250, loss=370.935388578658.
 niter=260, loss=370.39472188464026.
 niter=270, loss=369.9231339075728.
 niter=280,

In [10]:
# compile all packages' data 
package = (["sk_mu"])*6+(["sk_cd"])*6+(["nmf_torch_mu"])*6+(["nmf_torch_halsvar"])*6+(["nmf_torch_mu_cuda"])*6+(["nmf_torch_mu_10jobs"])*6

runtime_sec = sk_mu_s + sk_cd_s + nmf_torch_mu_s + nmf_torch_halsvar_s + nmf_torch_mu_cuda_s + nmf_torch_mu_10jobs_s
runtime_sec_log = np.log1p(runtime_sec)

memory = sk_mu_m + sk_cd_m + nmf_torch_mu_m + nmf_torch_halsvar_m + nmf_torch_mu_cuda_m + nmf_torch_mu_10jobs_m
memory_5300 = [x - 5000 for x in memory]

error = sk_mu_l + sk_cd_l + nmf_torch_mu_l + nmf_torch_halsvar_l + nmf_torch_mu_cuda_l + nmf_torch_mu_10jobs_l

In [ ]:
graph_benchmarking_results(package, runtime_sec, file_name, components = ([10,20,50,100,150,200])*6, fig_name = "Log Runtime(s)")
graph_benchmarking_results(package, memory_5300, file_name, components = ([10,20,50,100,150,200])*6, fig_name = "memory")
graph_benchmarking_results(package, error, file_name, components = ([10,20,50,100,150,200])*6, fig_name = "Explained Variance")